In [112]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_excel("Online Retail.xlsx")
# Typage
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], dayfirst=True, errors="coerce")
print(df.head(20))
print(f"\nTaille : {df.shape}")

   InvoiceNo StockCode                          Description  Quantity  \
0     536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1     536365     71053                  WHITE METAL LANTERN         6   
2     536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3     536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4     536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   
5     536365     22752         SET 7 BABUSHKA NESTING BOXES         2   
6     536365     21730    GLASS STAR FROSTED T-LIGHT HOLDER         6   
7     536366     22633               HAND WARMER UNION JACK         6   
8     536366     22632            HAND WARMER RED POLKA DOT         6   
9     536367     84879        ASSORTED COLOUR BIRD ORNAMENT        32   
10    536367     22745           POPPY'S PLAYHOUSE BEDROOM          6   
11    536367     22748            POPPY'S PLAYHOUSE KITCHEN         6   
12    536367     22749    FELTCRAFT PRINCESS CHARLO

In [113]:
df = df.query("'C' not in `InvoiceNo` and `Quantity` > 0 and `UnitPrice` > 0")
print(f"\nTaille : {df.shape}")


Taille : (530104, 8)


In [114]:
df["CustomerID"].isna().value_counts()

CustomerID
False    397884
True     132220
Name: count, dtype: int64

In [115]:
df.dropna(subset="CustomerID", inplace=True)
print(f"\nTaille : {df.shape}")


Taille : (397884, 8)


In [116]:
df.isna().value_counts()

InvoiceNo  StockCode  Description  Quantity  InvoiceDate  UnitPrice  CustomerID  Country
False      False      False        False     False        False      False       False      397884
Name: count, dtype: int64

In [117]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], dayfirst=True, errors="coerce")
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


In [118]:
df["montant_total_transaction"] = df["Quantity"] * df["UnitPrice"]

montant_total = df.groupby("CustomerID")["montant_total_transaction"].sum()
montant_total

CustomerID
12346.0    77183.60
12347.0     4310.00
12348.0     1797.24
12349.0     1757.55
12350.0      334.40
             ...   
18280.0      180.60
18281.0       80.82
18282.0      178.05
18283.0     2094.88
18287.0     1837.28
Name: montant_total_transaction, Length: 4338, dtype: float64

In [119]:
data = pd.DataFrame({"montant_total": montant_total})
data

,montant_total
CustomerID,
12346.0,77183.60
12347.0,4310.00
12348.0,1797.24
12349.0,1757.55
12350.0,334.40
...,...
18280.0,180.60
18281.0,80.82
18282.0,178.05


In [120]:
nb_tot_commande = df.groupby("CustomerID")["InvoiceNo"].nunique()
data["nb_tot_commande"] = nb_tot_commande
data

,montant_total,nb_tot_commande
CustomerID,,
12346.0,77183.60,1
12347.0,4310.00,7
12348.0,1797.24,4
12349.0,1757.55,1
12350.0,334.40,1
...,...,...
18280.0,180.60,1
18281.0,80.82,1
18282.0,178.05,2


In [121]:
data["panier_moyen"] = data["montant_total"] / data["nb_tot_commande"]
data

,montant_total,nb_tot_commande,panier_moyen
CustomerID,,,
12346.0,77183.60,1,77183.600000
12347.0,4310.00,7,615.714286
12348.0,1797.24,4,449.310000
12349.0,1757.55,1,1757.550000
12350.0,334.40,1,334.400000
...,...,...,...
18280.0,180.60,1,180.600000
18281.0,80.82,1,80.820000
18282.0,178.05,2,89.025000


In [122]:
data["depense_median_par_panier"] = (
    df.groupby(["CustomerID", "InvoiceNo"])["montant_total_transaction"]
    .sum()
    .groupby("CustomerID")
    .median()
)
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier
CustomerID,,,,
12346.0,77183.60,1,77183.600000,77183.600
12347.0,4310.00,7,615.714286,584.910
12348.0,1797.24,4,449.310000,338.500
12349.0,1757.55,1,1757.550000,1757.550
12350.0,334.40,1,334.400000,334.400
...,...,...,...,...
18280.0,180.60,1,180.600000,180.600
18281.0,80.82,1,80.820000,80.820
18282.0,178.05,2,89.025000,89.025


In [123]:
data["quantite_tot"] = df.groupby("CustomerID")["Quantity"].sum()
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot
CustomerID,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215
12347.0,4310.00,7,615.714286,584.910,2458
12348.0,1797.24,4,449.310000,338.500,2341
12349.0,1757.55,1,1757.550000,1757.550,631
12350.0,334.40,1,334.400000,334.400,197
...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45
18281.0,80.82,1,80.820000,80.820,54
18282.0,178.05,2,89.025000,89.025,103


In [124]:
data["Quantite_moyenne_par_commande"] = data["quantite_tot"] / data["nb_tot_commande"]
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande
CustomerID,,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215,74215.000000
12347.0,4310.00,7,615.714286,584.910,2458,351.142857
12348.0,1797.24,4,449.310000,338.500,2341,585.250000
12349.0,1757.55,1,1757.550000,1757.550,631,631.000000
12350.0,334.40,1,334.400000,334.400,197,197.000000
...,...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45,45.000000
18281.0,80.82,1,80.820000,80.820,54,54.000000
18282.0,178.05,2,89.025000,89.025,103,51.500000


In [125]:
from datetime import timedelta

recence = (
    df["InvoiceDate"].max()
    - df.groupby("CustomerID")["InvoiceDate"].max()
    + timedelta(days=1)
).dt.total_seconds()
data["recence"] = recence
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence
CustomerID,,,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215,74215.000000,28176540.0
12347.0,4310.00,7,615.714286,584.910,2458,351.142857,248280.0
12348.0,1797.24,4,449.310000,338.500,2341,585.250000,6565020.0
12349.0,1757.55,1,1757.550000,1757.550,631,631.000000,1652340.0
12350.0,334.40,1,334.400000,334.400,197,197.000000,26858940.0
...,...,...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45,45.000000,24029880.0
18281.0,80.82,1,80.820000,80.820,54,54.000000,15645420.0
18282.0,178.05,2,89.025000,89.025,103,51.500000,695220.0


In [126]:
anciennete_client = (
    df["InvoiceDate"].max()
    - df.groupby("CustomerID")["InvoiceDate"].min()
    + timedelta(days=1)
).dt.total_seconds()
data["anciennete_client"] = anciennete_client
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence,anciennete_client
CustomerID,,,,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215,74215.000000,28176540.0,28176540.0
12347.0,4310.00,7,615.714286,584.910,2458,351.142857,248280.0,31787580.0
12348.0,1797.24,4,449.310000,338.500,2341,585.250000,6565020.0,30994860.0
12349.0,1757.55,1,1757.550000,1757.550,631,631.000000,1652340.0,1652340.0
12350.0,334.40,1,334.400000,334.400,197,197.000000,26858940.0,26858940.0
...,...,...,...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45,45.000000,24029880.0,24029880.0
18281.0,80.82,1,80.820000,80.820,54,54.000000,15645420.0,15645420.0
18282.0,178.05,2,89.025000,89.025,103,51.500000,695220.0,10970100.0


In [ ]:
# Une date par commande
dates_commandes = (
    df.groupby(["CustomerID", "InvoiceNo"])["InvoiceDate"]
    .min()
    .reset_index()
    .sort_values(["CustomerID", "InvoiceDate"])
)

# Intervalle entre deux commandes successives
dates_commandes["intervalle_inter_achat"] = (
    dates_commandes.groupby("CustomerID")["InvoiceDate"].diff().dt.total_seconds()
)
intervalle_inter_achat_moyen = dates_commandes.groupby("CustomerID")[
    "intervalle_inter_achat"
].mean()

data["intervalle_inter_achat_moyen"] = intervalle_inter_achat_moyen
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence,anciennete_client,intervalle_inter_achat_moyen
CustomerID,,,,,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215,74215.000000,28176540.0,28176540.0,NaN
12347.0,4310.00,7,615.714286,584.910,2458,351.142857,248280.0,31787580.0,5256550.0
12348.0,1797.24,4,449.310000,338.500,2341,585.250000,6565020.0,30994860.0,8143280.0
12349.0,1757.55,1,1757.550000,1757.550,631,631.000000,1652340.0,1652340.0,NaN
12350.0,334.40,1,334.400000,334.400,197,197.000000,26858940.0,26858940.0,NaN
...,...,...,...,...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45,45.000000,24029880.0,24029880.0,NaN
18281.0,80.82,1,80.820000,80.820,54,54.000000,15645420.0,15645420.0,NaN
18282.0,178.05,2,89.025000,89.025,103,51.500000,695220.0,10970100.0,10274880.0


In [ ]:
intervalle_inter_achat_median = dates_commandes.groupby("CustomerID")[
    "intervalle_inter_achat"
].median()
data["intervalle_inter_achat_median"] = intervalle_inter_achat_median
data.fillna(0, inplace=True)
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence,anciennete_client,intervalle_inter_achat_moyen,intervalle_inter_achat_median
CustomerID,,,,,,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215,74215.000000,28176540.0,28176540.0,0.0,0.0
12347.0,4310.00,7,615.714286,584.910,2458,351.142857,248280.0,31787580.0,5256550.0,5050950.0
12348.0,1797.24,4,449.310000,338.500,2341,585.250000,6565020.0,30994860.0,8143280.0,6048300.0
12349.0,1757.55,1,1757.550000,1757.550,631,631.000000,1652340.0,1652340.0,0.0,0.0
12350.0,334.40,1,334.400000,334.400,197,197.000000,26858940.0,26858940.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45,45.000000,24029880.0,24029880.0,0.0,0.0
18281.0,80.82,1,80.820000,80.820,54,54.000000,15645420.0,15645420.0,0.0,0.0
18282.0,178.05,2,89.025000,89.025,103,51.500000,695220.0,10970100.0,10274880.0,10274880.0


In [ ]:
references_par_panier = df.groupby(["CustomerID", "InvoiceNo"])["StockCode"].nunique()

references_moyennes = references_par_panier.groupby("CustomerID").mean()

data["references_moyennes"] = references_moyennes
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence,anciennete_client,intervalle_inter_achat_moyen,intervalle_inter_achat_median,references_moyennes
CustomerID,,,,,,,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215,74215.000000,28176540.0,28176540.0,0.0,0.0,1.000000
12347.0,4310.00,7,615.714286,584.910,2458,351.142857,248280.0,31787580.0,5256550.0,5050950.0,26.000000
12348.0,1797.24,4,449.310000,338.500,2341,585.250000,6565020.0,30994860.0,8143280.0,6048300.0,6.750000
12349.0,1757.55,1,1757.550000,1757.550,631,631.000000,1652340.0,1652340.0,0.0,0.0,73.000000
12350.0,334.40,1,334.400000,334.400,197,197.000000,26858940.0,26858940.0,0.0,0.0,17.000000
...,...,...,...,...,...,...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45,45.000000,24029880.0,24029880.0,0.0,0.0,10.000000
18281.0,80.82,1,80.820000,80.820,54,54.000000,15645420.0,15645420.0,0.0,0.0,7.000000
18282.0,178.05,2,89.025000,89.025,103,51.500000,695220.0,10970100.0,10274880.0,10274880.0,6.000000


In [ ]:
references_moyennes = references_par_panier.groupby("CustomerID").median()

data["references_median"] = references_moyennes
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence,anciennete_client,intervalle_inter_achat_moyen,intervalle_inter_achat_median,references_moyennes,references_median
CustomerID,,,,,,,,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215,74215.000000,28176540.0,28176540.0,0.0,0.0,1.000000,1.0
12347.0,4310.00,7,615.714286,584.910,2458,351.142857,248280.0,31787580.0,5256550.0,5050950.0,26.000000,24.0
12348.0,1797.24,4,449.310000,338.500,2341,585.250000,6565020.0,30994860.0,8143280.0,6048300.0,6.750000,5.5
12349.0,1757.55,1,1757.550000,1757.550,631,631.000000,1652340.0,1652340.0,0.0,0.0,73.000000,73.0
12350.0,334.40,1,334.400000,334.400,197,197.000000,26858940.0,26858940.0,0.0,0.0,17.000000,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45,45.000000,24029880.0,24029880.0,0.0,0.0,10.000000,10.0
18281.0,80.82,1,80.820000,80.820,54,54.000000,15645420.0,15645420.0,0.0,0.0,7.000000,7.0
18282.0,178.05,2,89.025000,89.025,103,51.500000,695220.0,10970100.0,10274880.0,10274880.0,6.000000,6.0


In [ ]:
nb_ligne_par_panier = (
    df.groupby(["CustomerID", "InvoiceNo"]).size().groupby("CustomerID").mean()
)

data["nb_ligne_par_panier"] = nb_ligne_par_panier
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence,anciennete_client,intervalle_inter_achat_moyen,intervalle_inter_achat_median,references_moyennes,references_median,nb_ligne_par_panier
CustomerID,,,,,,,,,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215,74215.000000,28176540.0,28176540.0,0.0,0.0,1.000000,1.0,1.000000
12347.0,4310.00,7,615.714286,584.910,2458,351.142857,248280.0,31787580.0,5256550.0,5050950.0,26.000000,24.0,26.000000
12348.0,1797.24,4,449.310000,338.500,2341,585.250000,6565020.0,30994860.0,8143280.0,6048300.0,6.750000,5.5,7.750000
12349.0,1757.55,1,1757.550000,1757.550,631,631.000000,1652340.0,1652340.0,0.0,0.0,73.000000,73.0,73.000000
12350.0,334.40,1,334.400000,334.400,197,197.000000,26858940.0,26858940.0,0.0,0.0,17.000000,17.0,17.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45,45.000000,24029880.0,24029880.0,0.0,0.0,10.000000,10.0,10.000000
18281.0,80.82,1,80.820000,80.820,54,54.000000,15645420.0,15645420.0,0.0,0.0,7.000000,7.0,7.000000
18282.0,178.05,2,89.025000,89.025,103,51.500000,695220.0,10970100.0,10274880.0,10274880.0,6.000000,6.0,6.000000


In [ ]:
data["nb_moy__unite_par_panier"] = (
    df.groupby(["CustomerID", "InvoiceNo"])["Quantity"]
    .sum()
    .groupby("CustomerID")
    .mean()
)
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence,anciennete_client,intervalle_inter_achat_moyen,intervalle_inter_achat_median,references_moyennes,references_median,nb_ligne_par_panier,nb_moy__unite_par_panier
CustomerID,,,,,,,,,,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215,74215.000000,28176540.0,28176540.0,0.0,0.0,1.000000,1.0,1.000000,74215.000000
12347.0,4310.00,7,615.714286,584.910,2458,351.142857,248280.0,31787580.0,5256550.0,5050950.0,26.000000,24.0,26.000000,351.142857
12348.0,1797.24,4,449.310000,338.500,2341,585.250000,6565020.0,30994860.0,8143280.0,6048300.0,6.750000,5.5,7.750000,585.250000
12349.0,1757.55,1,1757.550000,1757.550,631,631.000000,1652340.0,1652340.0,0.0,0.0,73.000000,73.0,73.000000,631.000000
12350.0,334.40,1,334.400000,334.400,197,197.000000,26858940.0,26858940.0,0.0,0.0,17.000000,17.0,17.000000,197.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45,45.000000,24029880.0,24029880.0,0.0,0.0,10.000000,10.0,10.000000,45.000000
18281.0,80.82,1,80.820000,80.820,54,54.000000,15645420.0,15645420.0,0.0,0.0,7.000000,7.0,7.000000,54.000000
18282.0,178.05,2,89.025000,89.025,103,51.500000,695220.0,10970100.0,10274880.0,10274880.0,6.000000,6.0,6.000000,51.500000


In [133]:
nb_produits_distincts = df.groupby("CustomerID")["StockCode"].nunique()

data["nb_produits_distincts"] = nb_produits_distincts
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence,anciennete_client,intervalle_inter_achat_moyen,intervalle_inter_achat_median,references_moyennes,references_median,nb_ligne_par_panier,nb_moy__unite_par_panier,nb_produits_distincts
CustomerID,,,,,,,,,,,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215,74215.000000,28176540.0,28176540.0,0.0,0.0,1.000000,1.0,1.000000,74215.000000,1
12347.0,4310.00,7,615.714286,584.910,2458,351.142857,248280.0,31787580.0,5256550.0,5050950.0,26.000000,24.0,26.000000,351.142857,103
12348.0,1797.24,4,449.310000,338.500,2341,585.250000,6565020.0,30994860.0,8143280.0,6048300.0,6.750000,5.5,7.750000,585.250000,22
12349.0,1757.55,1,1757.550000,1757.550,631,631.000000,1652340.0,1652340.0,0.0,0.0,73.000000,73.0,73.000000,631.000000,73
12350.0,334.40,1,334.400000,334.400,197,197.000000,26858940.0,26858940.0,0.0,0.0,17.000000,17.0,17.000000,197.000000,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45,45.000000,24029880.0,24029880.0,0.0,0.0,10.000000,10.0,10.000000,45.000000,10
18281.0,80.82,1,80.820000,80.820,54,54.000000,15645420.0,15645420.0,0.0,0.0,7.000000,7.0,7.000000,54.000000,7
18282.0,178.05,2,89.025000,89.025,103,51.500000,695220.0,10970100.0,10274880.0,10274880.0,6.000000,6.0,6.000000,51.500000,12


In [ ]:
ca_par_produit = df.groupby(["CustomerID", "StockCode"])[
    "montant_total_transaction"
].sum()

part_ca_produit = ca_par_produit / ca_par_produit.groupby("CustomerID").transform("sum")

data["part_ca_top1"] = part_ca_produit.groupby("CustomerID").max()

data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence,anciennete_client,intervalle_inter_achat_moyen,intervalle_inter_achat_median,references_moyennes,references_median,nb_ligne_par_panier,nb_moy__unite_par_panier,nb_produits_distincts,part_ca_top1
CustomerID,,,,,,,,,,,,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215,74215.000000,28176540.0,28176540.0,0.0,0.0,1.000000,1.0,1.000000,74215.000000,1,1.000000
12347.0,4310.00,7,615.714286,584.910,2458,351.142857,248280.0,31787580.0,5256550.0,5050950.0,26.000000,24.0,26.000000,351.142857,103,0.086241
12348.0,1797.24,4,449.310000,338.500,2341,585.250000,6565020.0,30994860.0,8143280.0,6048300.0,6.750000,5.5,7.750000,585.250000,22,0.200307
12349.0,1757.55,1,1757.550000,1757.550,631,631.000000,1652340.0,1652340.0,0.0,0.0,73.000000,73.0,73.000000,631.000000,73,0.170692
12350.0,334.40,1,334.400000,334.400,197,197.000000,26858940.0,26858940.0,0.0,0.0,17.000000,17.0,17.000000,197.000000,17,0.119617
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45,45.000000,24029880.0,24029880.0,0.0,0.0,10.000000,10.0,10.000000,45.000000,10,0.131229
18281.0,80.82,1,80.820000,80.820,54,54.000000,15645420.0,15645420.0,0.0,0.0,7.000000,7.0,7.000000,54.000000,7,0.209725
18282.0,178.05,2,89.025000,89.025,103,51.500000,695220.0,10970100.0,10274880.0,10274880.0,6.000000,6.0,6.000000,51.500000,12,0.143218


In [ ]:
data["part_ca_top5"] = (
    part_ca_produit.sort_values(ascending=False)
    .groupby("CustomerID")
    .head(5)
    .groupby("CustomerID")
    .sum()
)
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence,anciennete_client,intervalle_inter_achat_moyen,intervalle_inter_achat_median,references_moyennes,references_median,nb_ligne_par_panier,nb_moy__unite_par_panier,nb_produits_distincts,part_ca_top1,part_ca_top5
CustomerID,,,,,,,,,,,,,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215,74215.000000,28176540.0,28176540.0,0.0,0.0,1.000000,1.0,1.000000,74215.000000,1,1.000000,1.000000
12347.0,4310.00,7,615.714286,584.910,2458,351.142857,248280.0,31787580.0,5256550.0,5050950.0,26.000000,24.0,26.000000,351.142857,103,0.086241,0.255543
12348.0,1797.24,4,449.310000,338.500,2341,585.250000,6565020.0,30994860.0,8143280.0,6048300.0,6.750000,5.5,7.750000,585.250000,22,0.200307,0.581113
12349.0,1757.55,1,1757.550000,1757.550,631,631.000000,1652340.0,1652340.0,0.0,0.0,73.000000,73.0,73.000000,631.000000,73,0.170692,0.297118
12350.0,334.40,1,334.400000,334.400,197,197.000000,26858940.0,26858940.0,0.0,0.0,17.000000,17.0,17.000000,197.000000,17,0.119617,0.421053
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45,45.000000,24029880.0,24029880.0,0.0,0.0,10.000000,10.0,10.000000,45.000000,10,0.131229,0.568937
18281.0,80.82,1,80.820000,80.820,54,54.000000,15645420.0,15645420.0,0.0,0.0,7.000000,7.0,7.000000,54.000000,7,0.209725,0.875278
18282.0,178.05,2,89.025000,89.025,103,51.500000,695220.0,10970100.0,10274880.0,10274880.0,6.000000,6.0,6.000000,51.500000,12,0.143218,0.564167


In [ ]:
import numpy as np

data["entropie_produit"] = (
    -(
        part_ca_produit[part_ca_produit > 0]
        * np.log1p(part_ca_produit[part_ca_produit > 0])
    )
    .groupby("CustomerID")
    .sum()
)

data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence,anciennete_client,intervalle_inter_achat_moyen,intervalle_inter_achat_median,references_moyennes,references_median,nb_ligne_par_panier,nb_moy__unite_par_panier,nb_produits_distincts,part_ca_top1,part_ca_top5,entropie_produit
CustomerID,,,,,,,,,,,,,,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215,74215.000000,28176540.0,28176540.0,0.0,0.0,1.000000,1.0,1.000000,74215.000000,1,1.000000,1.000000,-0.693147
12347.0,4310.00,7,615.714286,584.910,2458,351.142857,248280.0,31787580.0,5256550.0,5050950.0,26.000000,24.0,26.000000,351.142857,103,0.086241,0.255543,-0.023410
12348.0,1797.24,4,449.310000,338.500,2341,585.250000,6565020.0,30994860.0,8143280.0,6048300.0,6.750000,5.5,7.750000,585.250000,22,0.200307,0.581113,-0.088571
12349.0,1757.55,1,1757.550000,1757.550,631,631.000000,1652340.0,1652340.0,0.0,0.0,73.000000,73.0,73.000000,631.000000,73,0.170692,0.297118,-0.039333
12350.0,334.40,1,334.400000,334.400,197,197.000000,26858940.0,26858940.0,0.0,0.0,17.000000,17.0,17.000000,197.000000,17,0.119617,0.421053,-0.064043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45,45.000000,24029880.0,24029880.0,0.0,0.0,10.000000,10.0,10.000000,45.000000,10,0.131229,0.568937,-0.098031
18281.0,80.82,1,80.820000,80.820,54,54.000000,15645420.0,15645420.0,0.0,0.0,7.000000,7.0,7.000000,54.000000,7,0.209725,0.875278,-0.161927
18282.0,178.05,2,89.025000,89.025,103,51.500000,695220.0,10970100.0,10274880.0,10274880.0,6.000000,6.0,6.000000,51.500000,12,0.143218,0.564167,-0.090583


In [137]:
date_fin = df["InvoiceDate"].max()
date_debut_recente = date_fin - pd.Timedelta(days=90)
date_debut_historique = df["InvoiceDate"].min()

# Nombre de commandes récentes par client
nb_commandes_recentes = (
    df.loc[df["InvoiceDate"] >= date_debut_recente]
    .groupby("CustomerID")["InvoiceNo"]
    .nunique()
)

# Nombre de commandes historiques par client
nb_commandes_historiques = (
    df.loc[df["InvoiceDate"] < date_debut_recente]
    .groupby("CustomerID")["InvoiceNo"]
    .nunique()
)

duree_recente = 90
duree_historique = (date_debut_recente - date_debut_historique).days

activite_recente = nb_commandes_recentes / duree_recente
activite_historique = nb_commandes_historiques / duree_historique

ratio_activite_recente_historique = (
    activite_recente.div(activite_historique)
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .rename("ratio_activite_recente_historique")
)

data["ratio_activite_recente_historique"] = ratio_activite_recente_historique
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence,anciennete_client,intervalle_inter_achat_moyen,intervalle_inter_achat_median,references_moyennes,references_median,nb_ligne_par_panier,nb_moy__unite_par_panier,nb_produits_distincts,part_ca_top1,part_ca_top5,entropie_produit,ratio_activite_recente_historique
CustomerID,,,,,,,,,,,,,,,,,,,
12346.0,77183.60,1,77183.600000,77183.600,74215,74215.000000,28176540.0,28176540.0,0.0,0.0,1.000000,1.0,1.000000,74215.000000,1,1.000000,1.000000,-0.693147,0.000000
12347.0,4310.00,7,615.714286,584.910,2458,351.142857,248280.0,31787580.0,5256550.0,5050950.0,26.000000,24.0,26.000000,351.142857,103,0.086241,0.255543,-0.023410,1.257778
12348.0,1797.24,4,449.310000,338.500,2341,585.250000,6565020.0,30994860.0,8143280.0,6048300.0,6.750000,5.5,7.750000,585.250000,22,0.200307,0.581113,-0.088571,1.048148
12349.0,1757.55,1,1757.550000,1757.550,631,631.000000,1652340.0,1652340.0,0.0,0.0,73.000000,73.0,73.000000,631.000000,73,0.170692,0.297118,-0.039333,0.000000
12350.0,334.40,1,334.400000,334.400,197,197.000000,26858940.0,26858940.0,0.0,0.0,17.000000,17.0,17.000000,197.000000,17,0.119617,0.421053,-0.064043,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18280.0,180.60,1,180.600000,180.600,45,45.000000,24029880.0,24029880.0,0.0,0.0,10.000000,10.0,10.000000,45.000000,10,0.131229,0.568937,-0.098031,0.000000
18281.0,80.82,1,80.820000,80.820,54,54.000000,15645420.0,15645420.0,0.0,0.0,7.000000,7.0,7.000000,54.000000,7,0.209725,0.875278,-0.161927,0.000000
18282.0,178.05,2,89.025000,89.025,103,51.500000,695220.0,10970100.0,10274880.0,10274880.0,6.000000,6.0,6.000000,51.500000,12,0.143218,0.564167,-0.090583,3.144444


In [138]:
import numpy as np

data[data.select_dtypes(include="number").columns] = np.log1p(
    data[data.select_dtypes(include="number").columns]
)
data

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence,anciennete_client,intervalle_inter_achat_moyen,intervalle_inter_achat_median,references_moyennes,references_median,nb_ligne_par_panier,nb_moy__unite_par_panier,nb_produits_distincts,part_ca_top1,part_ca_top5,entropie_produit,ratio_activite_recente_historique
CustomerID,,,,,,,,,,,,,,,,,,,
12346.0,11.253955,0.693147,11.253955,11.253955,11.214735,11.214735,17.154000,17.154000,0.000000,0.000000,0.693147,0.693147,0.693147,11.214735,0.693147,0.693147,0.693147,-1.181387,0.000000
12347.0,8.368925,2.079442,6.424406,6.373166,7.807510,5.864037,12.422316,17.274586,15.474986,15.435087,3.295837,3.218876,3.295837,5.864037,4.644391,0.082723,0.227568,-0.023689,0.814381
12348.0,7.494564,1.609438,6.109936,5.827474,7.758761,6.373746,15.697266,17.249332,15.912704,15.615288,2.047693,1.871802,2.169054,6.373746,3.135494,0.182577,0.458129,-0.092742,0.716936
12349.0,7.472245,0.693147,7.472245,7.472245,6.448889,6.448889,14.317704,14.317704,0.000000,0.000000,4.304065,4.304065,4.304065,6.448889,4.304065,0.157595,0.260145,-0.040127,0.000000
12350.0,5.815324,0.693147,5.815324,5.815324,5.288267,5.288267,17.106109,17.106109,0.000000,0.000000,2.890372,2.890372,2.890372,5.288267,2.890372,0.112987,0.351398,-0.066186,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18280.0,5.201806,0.693147,5.201806,5.201806,3.828641,3.828641,16.994809,16.994809,0.000000,0.000000,2.397895,2.397895,2.397895,3.828641,2.397895,0.123305,0.450398,-0.103175,0.000000
18281.0,4.404522,0.693147,4.404522,4.404522,4.007333,4.007333,16.565689,16.565689,0.000000,0.000000,2.079442,2.079442,2.079442,4.007333,2.079442,0.190393,0.628757,-0.176651,0.000000
18282.0,5.187665,1.098612,4.500087,4.500087,4.644391,3.960813,13.451985,16.210684,16.145213,16.145213,1.945910,1.945910,1.945910,3.960813,2.564949,0.133847,0.447354,-0.094951,1.421769


In [139]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn import set_config

set_config(transform_output="pandas")

encoder = ColumnTransformer(
    [("encoder", OneHotEncoder(sparse_output=False), ["country"])],
    verbose_feature_names_out=False,
    remainder="passthrough",
)
pipeline = make_pipeline(StandardScaler())
data_encoder = pipeline.fit_transform(data)
data_encoder

,montant_total,nb_tot_commande,panier_moyen,depense_median_par_panier,quantite_tot,Quantite_moyenne_par_commande,recence,anciennete_client,intervalle_inter_achat_moyen,intervalle_inter_achat_median,references_moyennes,references_median,nb_ligne_par_panier,nb_moy__unite_par_panier,nb_produits_distincts,part_ca_top1,part_ca_top5,entropie_produit,ratio_activite_recente_historique
CustomerID,,,,,,,,,,,,,,,,,,,
12346.0,3.706225,-0.955214,7.523483,7.548127,3.816653,6.828298,1.436033,0.716043,-1.357200,-1.350964,-2.625675,-2.492447,-2.599584,6.828298,-2.526376,3.942818,1.988633,-5.384200,-0.764785
12347.0,1.411843,1.074425,1.038408,1.024832,1.329485,0.905794,-1.984457,0.851034,0.760407,0.776446,0.595886,0.545562,0.565912,0.905794,0.965881,-0.490767,-0.693671,0.441586,0.444036
12348.0,0.716489,0.386304,0.616141,0.295501,1.293900,1.469974,0.382974,0.822763,0.820305,0.801283,-0.949044,-1.074732,-0.804527,1.469974,-0.367738,0.234486,0.634643,0.094097,0.299394
12349.0,0.698739,-0.955214,2.445437,2.493778,0.337734,1.553147,-0.614299,-2.459063,-1.357200,-1.350964,1.843852,1.850855,1.792160,1.553147,0.665088,0.053036,-0.505988,0.358863,-0.764785
12350.0,-0.618962,-0.955214,0.220538,0.279262,-0.509484,0.268494,1.401413,0.662432,-1.357200,-1.350964,0.094009,0.150429,0.072769,0.268494,-0.584387,-0.270960,0.019740,0.227729,-0.764785
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18280.0,-1.106875,-0.955214,-0.603287,-0.540719,-1.574965,-1.347116,1.320955,0.537835,-1.357200,-1.350964,-0.515570,-0.441934,-0.526200,-1.347116,-1.019656,-0.196019,0.590103,0.041595,-0.764785
18281.0,-1.740933,-0.955214,-1.673874,-1.606310,-1.444525,-1.149328,1.010748,0.057455,-1.357200,-1.350964,-0.909746,-0.824978,-0.913517,-1.149328,-1.301117,0.291253,1.617668,-0.328152,-0.764785
18282.0,-1.118121,-0.361583,-1.545549,-1.478584,-0.979493,-1.200819,-1.240119,-0.339957,0.852121,0.874323,-1.075028,-0.985593,-1.075923,-1.200819,-0.872007,-0.119448,0.572562,0.082976,1.345608


In [142]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn import set_config

set_config(transform_output="pandas")
data_pca = PCA(random_state=42, n_components=3).fit_transform(data)
data_pca

,pca0,pca1,pca2
CustomerID,,,
12346.0,-13.178581,9.307831,-2.406584
12347.0,8.419207,2.748927,-1.507263
12348.0,8.468630,0.436967,-0.159562
12349.0,-13.644804,5.355254,-0.338487
12350.0,-14.059878,0.591987,1.176858
...,...,...,...
18280.0,-14.289313,-2.027651,0.802894
18281.0,-14.356186,-2.791506,0.115577
18282.0,8.588169,-3.955588,-1.376796


In [143]:
import plotly.express as px

fig = px.scatter_3d(
    data_pca, x=data_pca["pca0"], y=data_pca["pca1"], z=data_pca["pca2"]
)
fig.update_traces(marker_size=3)
fig.show()